# 🔬 Notebook 3: S3 (Object Storage) — Deep Dive

## 🛠️ Setup

```bash
cd 06-system-designs/s3
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Deep dive 1

### Presigned URLs

A presigned URL is a normal URL with an HMAC signature and an expiry. Anyone with the URL can make that one request without having your credentials.

The signature is `HMAC(secret, method|path|expiry)` — the server re-computes it and compares.

In [ ]:
import hmac, hashlib, time, urllib.parse

SECRET = b"super-secret"

def presign(method, bucket, key, ttl=60):
    expires = int(time.time()) + ttl
    msg = f"{method}|{bucket}|{key}|{expires}".encode()
    sig = hmac.new(SECRET, msg, hashlib.sha256).hexdigest()
    return f"/{bucket}/{key}?exp={expires}&sig={sig}"

def verify(method, bucket, key, url):
    qs = urllib.parse.parse_qs(url.split('?',1)[1])
    exp = int(qs['exp'][0]); sig = qs['sig'][0]
    if time.time() > exp: return False, "expired"
    msg = f"{method}|{bucket}|{key}|{exp}".encode()
    expected = hmac.new(SECRET, msg, hashlib.sha256).hexdigest()
    return hmac.compare_digest(sig, expected), "ok"

url = presign("GET", "b", "x")
print("URL:", url)
print("verify:", verify("GET", "b", "x", url))
print("wrong method:", verify("PUT", "b", "x", url))

## Deep dive 2

### Erasure coding (intuition)

Instead of 3 full copies (3× cost), split data into `k` shards and compute `m` parity shards; any `k` of `k+m` shards can reconstruct the data.

Below: XOR parity (k=2, m=1). Lose any one shard and we recover it.

In [ ]:
def xor(a: bytes, b: bytes) -> bytes:
    return bytes(x ^ y for x,y in zip(a,b))

data = b"hello world!!!!"       # 15 bytes; pad to even
if len(data) % 2: data += b"\0"
half = len(data)//2
d1, d2 = data[:half], data[half:]
p = xor(d1, d2)                  # parity

# Simulate losing d2
recovered_d2 = xor(d1, p)
print("d1 =", d1); print("d2 =", d2); print("p  =", p)
print("recovered d2:", recovered_d2)
print("full:", d1 + recovered_d2)

## Closing thoughts

- **Separate metadata from data** — different access patterns, different DBs.
- **Erasure coding** beats replication for cold storage.
- **Presigned URLs** let you offload bandwidth to clients without leaking keys.